# Evaluation results

Reads the test-set metric tables written by `scripts/evaluate.py` (`outputs/test_metrics_*.csv`) and presents them: the aggregate as a mean±std table, per view as bar charts (bar = mean, whisker = std), and per subject as a mean±std table sorted by MAE. All metrics are on the normalised [0, 1] scale.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / 'config.py').exists():
    ROOT = ROOT.parent
import sys; sys.path.insert(0, str(ROOT))
import config

OUT = (ROOT / config.OUTPUT_DIR).resolve()
agg = pd.read_csv(OUT / 'test_metrics_aggregate.csv')
vp  = pd.read_csv(OUT / 'test_metrics_per_view&pose.csv')
subj = pd.read_csv(OUT / 'test_metrics_per_subject.csv')

def pm(mean, std):
    """Format a mean±std cell like the terminal output (0.0845±0.0210)."""
    return f'{mean:.4f}±{std:.4f}'

print('loaded:', OUT)

## Aggregate over the whole test set

MAE, MSE and SSIM as columns; each cell is `mean±std` across the 42 test images.

In [ ]:
agg_tbl = (agg.assign(value=agg.apply(lambda r: pm(r['mean'], r['std']), axis=1))
              .set_index('metric')['value']
              .reindex(['MAE', 'MSE', 'SSIM'])
              .to_frame().T)
agg_tbl.index = ['mean±std']
agg_tbl.columns.name = None
(agg_tbl.style
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left'),
                                         ('padding', '10px 20px'),
                                         ('font-size', '15px')]},
            {'selector': 'td', 'props': [('padding', '10px 20px'),
                                         ('font-size', '15px')]},
        ]))

## Per view

One panel per metric, one bar per view (front / left / right), aggregated over poses. The bar height is the **mean**; the **whisker shows the standard deviation** — how much the metric varies across the test images (a longer whisker means more variability). Exact mean ± std values are listed in the table beneath the figure. For SSIM higher is better, while for MAE/MSE lower is better. Pose had a negligible effect — neutral and smile differed by at most ~0.007 across views — so results are aggregated over pose; the full per-pose breakdown remains in `test_metrics_per_view&pose.csv`.

In [ ]:
from IPython.display import display

views = ['front', 'left', 'right']
metrics = [('mae', 'MAE'), ('mse', 'MSE'), ('ssim', 'SSIM')]
view_colors = {'front': '#5a4fcf', 'left': '#e07b39', 'right': '#3a9679'}

def cell_val(view, col):
    row = vp[(vp.view == view) & (vp.pose == 'all')]   # per-view value, aggregated over poses
    return float(row[col].iloc[0])

fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
x = np.arange(len(views))
for ax, (key, label) in zip(axes, metrics):
    means = [cell_val(v, f'{key}_mean') for v in views]
    stds  = [cell_val(v, f'{key}_std')  for v in views]
    ax.bar(x, means, 0.6, yerr=stds, capsize=4,
           color=[view_colors[v] for v in views], alpha=0.9,
           error_kw=dict(elinewidth=1.2, alpha=0.7))
    top = max(m + s for m, s in zip(means, stds))
    ax.set_xticks(x)
    ax.set_xticklabels([v.capitalize() for v in views])
    ax.set_title(label, loc='left', fontweight='bold')
    ax.set_ylabel('mean ± std')
    ax.set_ylim(0, top * 1.15)
    ax.grid(True, axis='y', alpha=0.3)

fig.suptitle('Per view, aggregated over pose  (bar = mean, whisker = standard deviation)',
             x=0.0, ha='left', y=1.03, fontweight='bold')
plt.tight_layout()
plt.show()

# exact mean ± std per view, beneath the figure
tbl = pd.DataFrame(
    {label: [pm(cell_val(v, f'{key}_mean'), cell_val(v, f'{key}_std')) for v in views]
     for key, label in metrics},
    index=[v.capitalize() for v in views],
).rename_axis('View').reset_index()
display(tbl.style
        .hide(axis='index')
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left'),
                                         ('padding', '8px 18px'), ('font-size', '14px')]},
            {'selector': 'td', 'props': [('padding', '8px 18px'), ('font-size', '14px')]},
        ]))

## Per subject

Each subject's six images (all views and poses) aggregated into `mean±std`, sorted by MAE (lowest error first). The display-permitted subjects (p012, p019, p027) sit among the lower-error subjects.

In [ ]:
sub_tbl = subj.copy()
for key, label in [('mae', 'MAE'), ('mse', 'MSE'), ('ssim', 'SSIM')]:
    sub_tbl[label] = sub_tbl.apply(lambda r: pm(r[f'{key}_mean'], r[f'{key}_std']), axis=1)
sub_tbl = sub_tbl.rename(columns={'subject_id': 'Subject'})[['Subject', 'MAE', 'MSE', 'SSIM']]
(sub_tbl.style
        .hide(axis='index')
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left'),
                                         ('padding', '10px 20px'),
                                         ('font-size', '15px')]},
            {'selector': 'td', 'props': [('padding', '10px 20px'),
                                         ('font-size', '15px')]},
        ]))